### Load Data


In [1]:
import pandas as pd
import numpy as np

file_path = "../data/raw/RDU_starting2015_UTC.csv"

raw = pd.read_csv(
    file_path,
    na_values=["M"],
    low_memory=False
)

print("Raw shape:", raw.shape)
display(raw.head())
raw.info()


Raw shape: (102505, 30)


,station,valid,tmpf,dwpf,relh,drct,sknt,p01i,alti,mslp,...,wxcodes,ice_accretion_1hr,ice_accretion_3hr,ice_accretion_6hr,peak_wind_gust,peak_wind_drct,peak_wind_time,feel,metar,snowdepth
0,RDU,2015-01-01 00:51,33.0,24.0,69.17,0.0,0.0,0.00,30.36,1028.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,33.08,KRDU 010051Z 00000KT 10SM SCT250 01/M04 A3036 ...,NaN
1,RDU,2015-01-01 01:51,32.0,24.0,72.24,0.0,0.0,0.00,30.37,1028.6,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,32.00,KRDU 010151Z 00000KT 10SM FEW250 00/M04 A3037 ...,NaN
2,RDU,2015-01-01 02:51,29.0,23.0,78.18,0.0,0.0,0.00,30.36,1028.4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.94,KRDU 010251Z 00000KT 10SM FEW250 M02/M05 A3036...,NaN
3,RDU,2015-01-01 03:51,29.0,23.0,78.18,0.0,0.0,0.00,30.36,1028.3,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.94,KRDU 010351Z 00000KT 10SM CLR M02/M05 A3036 RM...,NaN
4,RDU,2015-01-01 04:51,28.0,23.0,81.11,0.0,0.0,0.00,30.36,1028.2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,28.04,KRDU 010451Z 00000KT 10SM FEW100 M02/M05 A3036...,NaN


<class 'pandas.DataFrame'>
RangeIndex: 102505 entries, 0 to 102504
Data columns (total 30 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   station            102505 non-null  str    
 1   valid              102505 non-null  str    
 2   tmpf               102496 non-null  float64
 3   dwpf               102498 non-null  float64
 4   relh               102451 non-null  float64
 5   drct               95077 non-null   float64
 6   sknt               102382 non-null  float64
 7   p01i               102504 non-null  str    
 8   alti               102505 non-null  float64
 9   mslp               101378 non-null  float64
 10  vsby               102496 non-null  float64
 11  gust               10579 non-null   float64
 12  skyc1              102501 non-null  str    
 13  skyc2              58531 non-null   str    
 14  skyc3              27080 non-null   str    
 15  skyc4              6760 non-null    str    
 16  skyl1        

In [2]:
from pathlib import Path

# Parse timestamps for filtering without changing the original columns
timestamps = pd.to_datetime(raw["valid"], utc=True)

# Keep all records from January 1, 2021 onward
start = pd.Timestamp("2021-01-01 00:00", tz="UTC")
raw_2021 = raw.loc[timestamps >= start].copy()

# Save the subset in the raw data folder
output_path = Path("../data/raw/RDU_starting2021_UTC.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

raw_2021.to_csv(output_path, index=False, na_rep="M")

print("Saved to:", output_path.resolve())
print("Rows:", len(raw_2021))
print("Start:", timestamps.loc[raw_2021.index].min())
print("End:", timestamps.loc[raw_2021.index].max())


Saved to: /Users/cassieliu/Desktop/AIPI520/aipi520-project1-rdu-forecasting/data/raw/RDU_starting2021_UTC.csv
Rows: 49974
Start: 2021-01-01 00:51:00+00:00
End: 2026-09-17 03:51:00+00:00


### Data Cleaning

In [3]:
# ---------- Inspect duplicates without removing data ----------

data = raw.copy()
data["valid"] = pd.to_datetime(data["valid"], utc=True)
data["hour"] = data["valid"].dt.floor("h")

# 1. Identical rows: count extra copies
identical_duplicates = data.duplicated().sum()
print("Extra identical rows:", identical_duplicates)

# 2. Repeated exact timestamps
timestamp_mask = data.duplicated(subset=["valid"], keep=False)

print(
    "Timestamps with multiple records:",
    data.loc[timestamp_mask, "valid"].nunique()
)

display(
    data.loc[timestamp_mask].sort_values("valid")
)

# 3. Multiple observations within the same hour
hour_mask = data.duplicated(subset=["hour"], keep=False)

print(
    "Hours with multiple records:",
    data.loc[hour_mask, "hour"].nunique()
)

display(
    data.loc[hour_mask].sort_values("valid")
)

Extra identical rows: 0
Timestamps with multiple records: 2


,station,valid,tmpf,dwpf,relh,drct,sknt,p01i,alti,mslp,...,ice_accretion_1hr,ice_accretion_3hr,ice_accretion_6hr,peak_wind_gust,peak_wind_drct,peak_wind_time,feel,metar,snowdepth,hour
95560,RDU,2025-11-30 14:25:00+00:00,NaN,NaN,NaN,290.0,4.0,NaN,30.33,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KRDU 301425Z AUTO 29004KT 10SM OVC013 05/M03 A...,NaN,2025-11-30 14:00:00+00:00
95561,RDU,2025-11-30 14:25:00+00:00,41.0,27.0,57.13,290.0,4.0,0.00,30.33,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,37.93,KRDU 301425Z 29004KT 10SM OVC013 05/M03 A3033 ...,NaN,2025-11-30 14:00:00+00:00
96007,RDU,2025-12-19 07:35:00+00:00,NaN,NaN,NaN,170.0,10.0,0.38,29.67,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KRDU 190735Z AUTO 17010KT 5SM -RA BR FEW004 OV...,NaN,2025-12-19 07:00:00+00:00
96008,RDU,2025-12-19 07:35:00+00:00,60.0,59.0,96.50,170.0,12.0,0.38,29.67,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,60.00,KRDU 190735Z 17012KT 5SM -RA BR FEW004 OVC050 ...,NaN,2025-12-19 07:00:00+00:00


Hours with multiple records: 3


,station,valid,tmpf,dwpf,relh,drct,sknt,p01i,alti,mslp,...,ice_accretion_1hr,ice_accretion_3hr,ice_accretion_6hr,peak_wind_gust,peak_wind_drct,peak_wind_time,feel,metar,snowdepth,hour
92370,RDU,2025-07-20 09:31:00+00:00,77.0,73.0,87.51,220.0,4.0,0.00,29.96,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,77.00,KRDU 200931Z 22004KT 10SM SCT003 SCT100 25/23 ...,NaN,2025-07-20 09:00:00+00:00
92371,RDU,2025-07-20 09:51:00+00:00,77.0,73.0,87.51,230.0,3.0,0.00,29.96,1014.0,...,NaN,NaN,NaN,NaN,NaN,NaN,77.00,KRDU 200951Z 23003KT 10SM FEW003 FEW100 25/23 ...,NaN,2025-07-20 09:00:00+00:00
95560,RDU,2025-11-30 14:25:00+00:00,NaN,NaN,NaN,290.0,4.0,NaN,30.33,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KRDU 301425Z AUTO 29004KT 10SM OVC013 05/M03 A...,NaN,2025-11-30 14:00:00+00:00
95561,RDU,2025-11-30 14:25:00+00:00,41.0,27.0,57.13,290.0,4.0,0.00,30.33,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,37.93,KRDU 301425Z 29004KT 10SM OVC013 05/M03 A3033 ...,NaN,2025-11-30 14:00:00+00:00
96007,RDU,2025-12-19 07:35:00+00:00,NaN,NaN,NaN,170.0,10.0,0.38,29.67,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,KRDU 190735Z AUTO 17010KT 5SM -RA BR FEW004 OV...,NaN,2025-12-19 07:00:00+00:00
96008,RDU,2025-12-19 07:35:00+00:00,60.0,59.0,96.50,170.0,12.0,0.38,29.67,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,60.00,KRDU 190735Z 17012KT 5SM -RA BR FEW004 OVC050 ...,NaN,2025-12-19 07:00:00+00:00


In [4]:
# ---------- Inspect observation minutes ----------

# Count observations at each minute
print("Observation counts by minute:")
print(data["valid"].dt.minute.value_counts().sort_index())

# Find non-:51 observations
non_51 = data[data["valid"].dt.minute != 51].copy()

print("\nNon-:51 observations:", len(non_51))

# Identify hours that contain a :51 report
hours_with_51 = data.loc[
    data["valid"].dt.minute == 51, "hour"
].unique()

# Check whether each non-:51 observation has a :51 alternative
non_51["has_51_in_same_hour"] = non_51["hour"].isin(hours_with_51)

display(
    non_51[
        ["valid", "hour", "tmpf", "has_51_in_same_hour"]
    ].sort_values("valid")
)

Observation counts by minute:
valid
18         1
25         2
29         1
30         1
31         1
35         2
51    102497
Name: count, dtype: int64

Non-:51 observations: 8


,valid,hour,tmpf,has_51_in_same_hour
89359,2025-03-16 15:18:00+00:00,2025-03-16 15:00:00+00:00,72.0,False
92370,2025-07-20 09:31:00+00:00,2025-07-20 09:00:00+00:00,77.0,True
95560,2025-11-30 14:25:00+00:00,2025-11-30 14:00:00+00:00,NaN,False
95561,2025-11-30 14:25:00+00:00,2025-11-30 14:00:00+00:00,41.0,False
96007,2025-12-19 07:35:00+00:00,2025-12-19 07:00:00+00:00,NaN,False
96008,2025-12-19 07:35:00+00:00,2025-12-19 07:00:00+00:00,60.0,False
97132,2026-02-04 15:30:00+00:00,2026-02-04 15:00:00+00:00,41.0,False
101267,2026-07-27 13:29:00+00:00,2026-07-27 13:00:00+00:00,77.0,False


In [5]:
# ---------- Inspect missing data ----------

# 1. Reports exist, but their temperature is missing
missing_temperature = data[data["tmpf"].isna()]

print("Reports with missing temperature:", len(missing_temperature))
display(missing_temperature[["valid", "tmpf"]])

# 2. Create the expected hourly timeline
start = pd.Timestamp("2015-01-01 00:00", tz="UTC")
end = pd.Timestamp("2026-09-17 04:00:00", tz="UTC")

expected_hours = pd.date_range(
    start=start,
    end=end,
    freq="h",
    inclusive="left"
)

# 3. Hours with no report at any minute
reported_hours = pd.DatetimeIndex(data["hour"].unique())
missing_hours = expected_hours.difference(reported_hours)

print("Hours with no report:", len(missing_hours))
display(pd.DataFrame({"missing_hour": missing_hours}))

# 4. Hours with no :51 report
reported_51_hours = pd.DatetimeIndex(
    data.loc[data["valid"].dt.minute == 51, "hour"].unique()
)

missing_51_hours = expected_hours.difference(reported_51_hours)

print("Hours with no :51 report:", len(missing_51_hours))

Reports with missing temperature: 9


,valid,tmpf
38089,2019-05-09 12:51:00+00:00,NaN
54252,2021-03-13 19:51:00+00:00,NaN
54253,2021-03-13 20:51:00+00:00,NaN
54254,2021-03-13 21:51:00+00:00,NaN
54255,2021-03-13 22:51:00+00:00,NaN
68962,2022-11-17 06:51:00+00:00,NaN
89290,2025-03-13 18:51:00+00:00,NaN
95560,2025-11-30 14:25:00+00:00,NaN
96007,2025-12-19 07:35:00+00:00,NaN


Hours with no report: 150


,missing_hour
0,2015-01-21 08:00:00+00:00
1,2015-03-08 23:00:00+00:00
2,2015-03-15 16:00:00+00:00
3,2015-06-17 04:00:00+00:00
4,2015-06-17 05:00:00+00:00
...,...
145,2026-05-18 04:00:00+00:00
146,2026-05-18 05:00:00+00:00
147,2026-06-30 20:00:00+00:00
148,2026-06-30 21:00:00+00:00


Hours with no :51 report: 155


In [6]:
# ---------------------Check values that cannot be converted to numbers------------------
temperature = pd.to_numeric(data["tmpf"], errors="coerce")

invalid_mask = data["tmpf"].notna() & temperature.isna()

print("Non-numeric temperature values:", invalid_mask.sum())
display(data.loc[invalid_mask, ["valid", "tmpf"]])

# Inspect the distribution and extreme observations
display(temperature.describe())

display(
    data.assign(temp_numeric=temperature)
    .sort_values("temp_numeric")
    [["valid", "tmpf"]]
    .head(10)
)

display(
    data.assign(temp_numeric=temperature)
    .sort_values("temp_numeric", ascending=False)
    [["valid", "tmpf"]]
    .head(10)
)

Non-numeric temperature values: 0


,valid,tmpf


count    102496.000000
mean         62.451443
std          16.801779
min           4.000000
25%          50.000000
50%          65.000000
75%          75.000000
max         104.000000
Name: tmpf, dtype: float64

,valid,tmpf
26421,2018-01-07 11:51:00+00:00,4.0
26419,2018-01-07 09:51:00+00:00,6.0
26420,2018-01-07 10:51:00+00:00,6.0
26422,2018-01-07 12:51:00+00:00,6.0
1209,2015-02-20 10:51:00+00:00,7.0
1210,2015-02-20 11:51:00+00:00,7.0
1208,2015-02-20 09:51:00+00:00,8.0
26418,2018-01-07 08:51:00+00:00,8.1
1206,2015-02-20 07:51:00+00:00,9.0
17717,2017-01-09 12:51:00+00:00,9.0


,valid,tmpf
83267,2024-07-05 17:51:00+00:00,104.0
83268,2024-07-05 18:51:00+00:00,103.0
65764,2022-07-06 18:51:00+00:00,102.0
83293,2024-07-06 19:51:00+00:00,102.0
100195,2026-06-12 19:51:00+00:00,102.0
100194,2026-06-12 18:51:00+00:00,102.0
83054,2024-06-26 19:51:00+00:00,102.0
102182,2026-09-03 17:51:00+00:00,101.0
100196,2026-06-12 20:51:00+00:00,101.0
100698,2026-07-03 20:51:00+00:00,101.0


In [7]:
# ---------- Clean hourly temperature data ----------

data = raw.copy()

# 1. Parse timestamps and temperature
data["valid"] = pd.to_datetime(
    data["valid"], utc=True, errors="raise"
)
data["tmpf"] = pd.to_numeric(
    data["tmpf"], errors="coerce"
)

# 2. Keep RDU records within the agreed UTC range
start = pd.Timestamp("2015-01-01 00:00", tz="UTC")
end = pd.Timestamp("2026-09-17 04:00:00", tz="UTC")

data = data[
    (data["station"] == "RDU") &
    (data["valid"] >= start) &
    (data["valid"] < end)
].copy()

# 3. Count and remove identical duplicate rows
identical_duplicates = data.duplicated().sum()
data = data.drop_duplicates().copy()

# 4. Create hour labels and report repeated timestamps/hours
data["ds"] = data["valid"].dt.floor("h")

duplicate_timestamps = data.loc[
    data.duplicated(subset=["valid"], keep=False), "valid"
].nunique()

multiple_record_hours = data.loc[
    data.duplicated(subset=["ds"], keep=False), "ds"
].nunique()

# 5. Consider only observations with valid temperatures
usable = data.dropna(subset=["tmpf"]).copy()

# Stop if the same timestamp has conflicting valid temperatures
temperature_counts = usable.groupby("valid")["tmpf"].nunique()
conflicting_times = temperature_counts[
    temperature_counts > 1
].index

if len(conflicting_times) > 0:
    display(
        usable.loc[
            usable["valid"].isin(conflicting_times),
            ["valid", "tmpf"]
        ]
    )
    raise ValueError("Conflicting temperatures at the same timestamp.")

# 6. Rank observations by distance from :51
preferred_time = usable["ds"] + pd.Timedelta(minutes=51)

usable["distance_to_51"] = (
    usable["valid"] - preferred_time
).abs()

# Closest to :51 wins; ties prefer the later observation
usable = usable.sort_values(
    ["ds", "distance_to_51", "valid"],
    ascending=[True, True, False]
)

selected = usable.drop_duplicates(
    subset=["ds"], keep="first"
).copy()

selected_51 = (selected["valid"].dt.minute == 51).sum()
selected_other = len(selected) - selected_51

# 7. Build the complete expected hourly timeline
expected_hours = pd.date_range(
    start=start,
    end=end,
    freq="h",
    inclusive="left"
)

reported_hours = pd.DatetimeIndex(data["ds"].unique())
usable_hours = pd.DatetimeIndex(selected["ds"])

# Separate absent reports from reports without valid temperatures
no_report_hours = len(expected_hours.difference(reported_hours))
no_valid_temperature_hours = len(
    reported_hours.difference(usable_hours)
)

# 8. Keep ds/y and preserve missing hours as NaN
clean = (
    selected[["ds", "tmpf"]]
    .rename(columns={"tmpf": "y"})
    .set_index("ds")
    .reindex(expected_hours)
    .rename_axis("ds")
    .reset_index()
)

# 9. Report cleaning results
print("Identical duplicate rows removed:", identical_duplicates)
print("Repeated exact timestamps:", duplicate_timestamps)
print("Hours with multiple records:", multiple_record_hours)
print("Hours using :51 observations:", selected_51)
print("Hours using other-minute observations:", selected_other)
print("Hours without any report:", no_report_hours)
print("Reported hours without valid temperature:", no_valid_temperature_hours)
print("Total missing targets:", clean["y"].isna().sum())
print("Total hours:", len(clean))
print("Observed targets:", clean["y"].notna().sum())

# Inspect observations used as alternatives to :51
display(
    selected.loc[
        selected["valid"].dt.minute != 51,
        ["valid", "ds", "tmpf"]
    ].sort_values("ds")
)

display(clean.head())
display(clean.tail())

Identical duplicate rows removed: 0
Repeated exact timestamps: 2
Hours with multiple records: 3
Hours using :51 observations: 102490
Hours using other-minute observations: 5
Hours without any report: 150
Reported hours without valid temperature: 7
Total missing targets: 157
Total hours: 102652
Observed targets: 102495


,valid,ds,tmpf
89359,2025-03-16 15:18:00+00:00,2025-03-16 15:00:00+00:00,72.0
95561,2025-11-30 14:25:00+00:00,2025-11-30 14:00:00+00:00,41.0
96008,2025-12-19 07:35:00+00:00,2025-12-19 07:00:00+00:00,60.0
97132,2026-02-04 15:30:00+00:00,2026-02-04 15:00:00+00:00,41.0
101267,2026-07-27 13:29:00+00:00,2026-07-27 13:00:00+00:00,77.0


,ds,y
0,2015-01-01 00:00:00+00:00,33.0
1,2015-01-01 01:00:00+00:00,32.0
2,2015-01-01 02:00:00+00:00,29.0
3,2015-01-01 03:00:00+00:00,29.0
4,2015-01-01 04:00:00+00:00,28.0


,ds,y
102647,2026-09-16 23:00:00+00:00,78.0
102648,2026-09-17 00:00:00+00:00,76.0
102649,2026-09-17 01:00:00+00:00,72.0
102650,2026-09-17 02:00:00+00:00,71.0
102651,2026-09-17 03:00:00+00:00,70.0


# Splitting the data

In [8]:
# ---------- Split data chronologically ----------

# UTC boundaries; end times are exclusive
val_start = pd.Timestamp("2026-08-20 04:00:00", tz="UTC")
test_start = pd.Timestamp("2026-09-03 04:00:00", tz="UTC")
data_end = pd.Timestamp("2026-09-17 04:00:00", tz="UTC")

# Training: all hours before the validation window
train = clean[
    clean["ds"] < val_start
].copy()

# Validation: 14 days
val = clean[
    (clean["ds"] >= val_start) &
    (clean["ds"] < test_start)
].copy()

# Test: the following 14 days
test = clean[
    (clean["ds"] >= test_start) &
    (clean["ds"] < data_end)
].copy()

# Summarize each split
summary = pd.DataFrame({
    "Split": ["Train", "Validation", "Test"],
    "Start": [train["ds"].min(), val["ds"].min(), test["ds"].min()],
    "End": [train["ds"].max(), val["ds"].max(), test["ds"].max()],
    "Total hours": [len(train), len(val), len(test)],
    "Observed targets": [
        train["y"].notna().sum(),
        val["y"].notna().sum(),
        test["y"].notna().sum()
    ],
    "Missing targets": [
        train["y"].isna().sum(),
        val["y"].isna().sum(),
        test["y"].isna().sum()
    ]
})

display(summary)

# Check that both evaluation windows contain 336 hours
assert len(val) == 14 * 24, "Validation must contain 336 hours."
assert len(test) == 14 * 24, "Test must contain 336 hours."

# Check that every cleaned row belongs to one split
assert len(train) + len(val) + len(test) == len(clean)

,Split,Start,End,Total hours,Observed targets,Missing targets
0,Train,2015-01-01 00:00:00+00:00,2026-08-20 03:00:00+00:00,101980,101823,157
1,Validation,2026-08-20 04:00:00+00:00,2026-09-03 03:00:00+00:00,336,336,0
2,Test,2026-09-03 04:00:00+00:00,2026-09-17 03:00:00+00:00,336,336,0


### Basic Feature Engineering

In [9]:
# ---------- Shared feature engineering ----------

def add_time_features(frame):
    data = frame.copy()

    # Extract UTC calendar features
    data["hour"] = data["ds"].dt.hour
    data["month"] = data["ds"].dt.month
    data["dayofyear"] = data["ds"].dt.dayofyear

    # Encode the daily cycle
    data["hour_sin"] = np.sin(2 * np.pi * data["hour"] / 24)
    data["hour_cos"] = np.cos(2 * np.pi * data["hour"] / 24)

    # Calculate annual position, accounting for leap years
    days_in_year = np.where(
        data["ds"].dt.is_leap_year, 366, 365
    )

    annual_position = (
        data["dayofyear"] - 1 + data["hour"] / 24
    ) / days_in_year

    # Encode the annual cycle
    data["year_sin"] = np.sin(2 * np.pi * annual_position)
    data["year_cos"] = np.cos(2 * np.pi * annual_position)

    return data


# Apply the same feature rules to each split
train_fe = add_time_features(train)
val_fe = add_time_features(val)
test_fe = add_time_features(test)

display(train_fe.head())

,ds,y,hour,month,dayofyear,hour_sin,hour_cos,year_sin,year_cos
0,2015-01-01 00:00:00+00:00,33.0,0,1,1,0.000000,1.000000,0.000000,1.000000
1,2015-01-01 01:00:00+00:00,32.0,1,1,1,0.258819,0.965926,0.000717,1.000000
2,2015-01-01 02:00:00+00:00,29.0,2,1,1,0.500000,0.866025,0.001435,0.999999
3,2015-01-01 03:00:00+00:00,29.0,3,1,1,0.707107,0.707107,0.002152,0.999998
4,2015-01-01 04:00:00+00:00,28.0,4,1,1,0.866025,0.500000,0.002869,0.999996


In [10]:
from pathlib import Path

# Create the output folder if it does not exist
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

# Save the shared datasets
train_fe.to_csv(output_dir / "train.csv", index=False)
val_fe.to_csv(output_dir / "val.csv", index=False)
test_fe.to_csv(output_dir / "test.csv", index=False)
